# Simulation Week: Modeling Types & Monte Carlo Methods

**Goal for today:**
1. Understand the two big flavors of models: **deterministic** and **stochastic**
2. Go deep on **stochastic modeling** — why randomness shows up in math at all
3. Learn the **Monte Carlo method** and use it to solve two problems by simulation instead of algebra

## 1. Deterministic vs. Stochastic Models

A **model** is just a simplified version of reality, built with math, that lets us predict or explain something.

| | Deterministic | Stochastic |
|---|---|---|
| **Definition** | Same inputs → same output, every single time | Same inputs → output can change, because randomness is built in |
| **Uses** | Probability, statistics, chance | No randomness — exact equations |
| **Examples** | Newton's Method, RK4, projectile motion (no wind) | Dice games, stock prices, radioactive decay, weather, epidemics |
| **Question it answers** | "What *will* happen?" | "What is *likely* to happen, and how likely?" |

You've already worked with deterministic models (RK4, Newton's Method). Today we flip to the other side.

> **Quick check:** Is a coin flip deterministic or stochastic? What about the trajectory of a ball thrown in a vacuum?

## 2. Stochastic Modeling, in Detail

### Why bring randomness into math at all?
Real systems are full of uncertainty we can't (or don't want to) track exactly:
- We don't know *exactly* which way each air molecule will push a falling leaf
- We don't know *exactly* how many people an infected person will meet tomorrow
- We don't know *exactly* which way a stock will move tomorrow

Instead of tracking every detail, a stochastic model says: **treat the uncertain part as a random variable, drawn from a probability distribution**, and study the pattern of outcomes rather than one exact outcome.

### The idea that makes this work: Law of Large Numbers
If you repeat a random experiment many times and average the results, that average **converges** to the true expected value.

- Flip a coin 10 times → you might get 7 heads (70%)
- Flip a coin 10,000 times → you'll land very close to 50%

This single fact is the entire engine behind the Monte Carlo method.

### Families of stochastic models (just so you've seen the names)
- **Random walks** — a path where each step is random (stock prices, particle diffusion)
- **Markov chains** — the next state depends only on the current state, with random transitions
- **Monte Carlo simulation** — what we're doing today
- **Stochastic differential equations** — ODEs (like the ones you solved with RK4) with a random noise term added

We're focusing on Monte Carlo because it's the most direct way to *see* the Law of Large Numbers in action.

## 3. The Monte Carlo Method

**Named after** the Monte Carlo Casino in Monaco — coined by mathematician Stanislaw Ulam while working on the Manhattan Project in the 1940s, when he realized random sampling could solve problems that were too hard to compute exactly.

**Core idea:** Instead of solving a problem exactly with equations, *simulate it randomly, many times*, and let the Law of Large Numbers give you an answer that gets more accurate the more times you simulate.

**General recipe:**
1. Define the domain of possible random inputs
2. Generate random inputs from a probability distribution
3. Run a deterministic calculation on each random input
4. Aggregate all the results into an estimate

Let's use this recipe on two problems.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)  # seeded so results are reproducible in class

## 4. Activity 1 — Estimating π by Throwing Random Darts

**Setup:** Picture a circle of radius 1 inscribed inside a 2×2 square.

$$\text{Area of circle} = \pi r^2 = \pi \qquad \text{Area of square} = (2r)^2 = 4$$

$$\frac{\text{Area of circle}}{\text{Area of square}} = \frac{\pi}{4}$$

**The Monte Carlo trick:** if we throw random darts uniformly at the square, the *fraction* that land inside the circle should approximate that same ratio:

$$\pi \approx 4 \times \frac{\text{darts inside circle}}{\text{total darts thrown}}$$

No calculus, no formula for π — just counting random points.

In [ ]:
def estimate_pi(n_points, rng):
    x = rng.uniform(-1, 1, n_points)
    y = rng.uniform(-1, 1, n_points)
    inside = (x**2 + y**2) <= 1
    pi_estimate = 4 * np.sum(inside) / n_points
    return pi_estimate, x, y, inside

n_points = 2000
pi_est, x, y, inside = estimate_pi(n_points, rng)
print(f"Estimated pi with {n_points} darts: {pi_est}")
print(f"Actual pi: {np.pi}")
print(f"Error: {abs(pi_est - np.pi):.4f}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(x[inside], y[inside], s=4, color="crimson", label="inside circle")
plt.scatter(x[~inside], y[~inside], s=4, color="steelblue", label="outside circle")
plt.gca().set_aspect("equal")
plt.title(f"Monte Carlo π estimate: {pi_est:.4f} (n={n_points})")
plt.legend(loc="upper right")
plt.show()

### Watching the Law of Large Numbers happen

Let's rerun the estimate at increasing numbers of darts and plot how the error shrinks. This is the Law of Large Numbers made visible.

In [ ]:
sample_sizes = [10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]
errors = []

for n in sample_sizes:
    pi_est, *_ = estimate_pi(n, rng)
    errors.append(abs(pi_est - np.pi))

plt.figure(figsize=(7, 5))
plt.loglog(sample_sizes, errors, marker="o")
plt.xlabel("Number of random darts (n)")
plt.ylabel("Absolute error vs. true π")
plt.title("Monte Carlo error shrinks as n grows")
plt.grid(True, which="both", alpha=0.3)
plt.show()

**Discussion:** notice the error doesn't shrink smoothly — it's noisy but trends downward. Monte Carlo error famously shrinks proportional to $1/\sqrt{n}$, which is why you need *4×* as many samples just to cut your error in half.

## 5. Activity 2 — Monte Carlo for Probability: Rolling a 7

Here's a question you *can* solve exactly with algebra, so we can check our simulation against the true answer.

**Question:** If you roll two fair six-sided dice, what's the probability the sum is 7?

**Exact answer:** there are 36 equally likely (die1, die2) pairs, and 6 of them sum to 7: (1,6) (2,5) (3,4) (4,3) (5,2) (6,1).
$$P(\text{sum}=7) = \frac{6}{36} = \frac{1}{6} \approx 0.1667$$

**Monte Carlo approach:** instead of counting outcomes, just *roll the dice a huge number of times in code* and count how often we actually get a 7.

In [ ]:
n_rolls = 100000
die1 = rng.integers(1, 7, n_rolls)  # 1-6 inclusive
die2 = rng.integers(1, 7, n_rolls)
totals = die1 + die2

sim_prob = np.mean(totals == 7)
true_prob = 1 / 6

print(f"Simulated P(sum=7) over {n_rolls} rolls: {sim_prob:.4f}")
print(f"True P(sum=7): {true_prob:.4f}")
print(f"Error: {abs(sim_prob - true_prob):.4f}")

In [ ]:
values, counts = np.unique(totals, return_counts=True)
plt.figure(figsize=(8, 5))
plt.bar(values, counts / n_rolls, color="seagreen")
plt.xlabel("Sum of two dice")
plt.ylabel("Simulated probability")
plt.title(f"Distribution of dice sums over {n_rolls} simulated rolls")
plt.xticks(range(2, 13))
plt.show()

**Discussion:** the whole triangular shape of that bar chart — more ways to make 7 than to make 2 or 12 — fell out of the simulation for free, with zero combinatorics. That's the power of Monte Carlo: it can answer questions that would be genuinely hard to count by hand (try this with 5 dice instead of 2).

## 6. Key Takeaways

- **Deterministic models** give exact answers from exact inputs. **Stochastic models** embrace uncertainty using probability.
- The **Law of Large Numbers** guarantees that averaging many random trials converges to the true expected value — this is *why* Monte Carlo works at all.
- The **Monte Carlo method**: generate random inputs → run a deterministic calculation on each → aggregate the results into an estimate.
- Monte Carlo error shrinks proportional to $1/\sqrt{n}$ — more samples = better accuracy, but with diminishing returns.
- Monte Carlo shines when a problem is too complex to solve exactly (or you're just too lazy to do the combinatorics).

## 7. Try It Yourself (Exercises)

1. **Modify Activity 1** to estimate π using only the *upper-right quarter* of the circle (x, y both between 0 and 1) instead of the full circle. Does your estimate change?
2. **Extend Activity 2** to 3 dice. What's the most likely sum? Simulate it and check your intuition.
3. **New problem:** Use Monte Carlo to estimate the area under the curve $y = \sin(x)$ from $x=0$ to $x=\pi$. (Hint: use the same "random darts in a box" trick as Activity 1, but with a rectangle that fully contains the curve.)
4. **Challenge:** In Activity 1, how many darts do you need, roughly, to get 3 correct decimal digits of π reliably? Use the error-vs-n plot to estimate it.